# EmT reproduction on FACED
Run the authors original FACED classification pipeline. Their default code uses individual rating index 8 despite the default V label. This differs from fixed stimulus labels.

In [ ]:
import sys,subprocess,importlib.util
from pathlib import Path
art=Path(r'D:\EEG Experiment Artifacts\2026-11-02\graph_experiments')
source=art/'paper_sources'/'EmT'
expected_commit='3ea05dc90a915182ab3407191c287456f912316d'
if not source.exists():
    source.parent.mkdir(parents=True,exist_ok=True)
    subprocess.run(['git','clone','https://github.com/yi-ding-cs/EmT.git',str(source)],check=True)
    subprocess.run(['git','-C',str(source),'checkout','--detach',expected_commit],check=True)
actual=subprocess.check_output(['git','-C',str(source),'rev-parse','HEAD'],text=True).strip()
assert actual==expected_commit, 'Unexpected EmT source revision'
deps=art/'paper_deps'
sys.path.insert(0,str(deps))
if importlib.util.find_spec('einops') is None:
    subprocess.run([sys.executable,'-m','pip','install','--target',str(deps),'einops==0.8.1'],check=True)
print('Original EmT source:',actual)


In [ ]:
import os,sys,json,time,shutil,hashlib,zipfile,subprocess
from types import SimpleNamespace
from pathlib import Path
root=Path(r'D:\EEG Experiment Artifacts\2026-11-02\graph_experiments')
source=root/'paper_sources'/'EmT'/'classification';deps=root/'paper_deps'
sys.path[:0]=[str(deps),str(source)]
from datasets.FACED import FACED
import pickle
output=root/'emt_reproduction';data_root=output/'input';rating_root=data_root/'Rating';output.mkdir(parents=True,exist_ok=True)
zip_path=Path(r'C:\FACED dataset\Other Files\Processed_data.zip')
source_commit=subprocess.check_output(['git','-C',str(source.parent),'rev-parse','HEAD'],text=True).strip()
args=SimpleNamespace(ROOT=str(output),data_path=str(data_root),dataset='FACED',data_format='rPSD',label_type='V',graph_type='BL',sampling_rate=250,num_class=2,sub_segment=4,segment=20,overlap=.8,sub_overlap=.75)
markers=output/'input_markers';markers.mkdir(parents=True,exist_ok=True)
class FACEDZipInput(FACED):
    def process_data(self,subject_path):
        member='Processed_data/'+Path(subject_path).name
        with archive.open(member) as f:data=pickle.load(f)
        return data.transpose(2,1,0)
started=time.perf_counter()
with zipfile.ZipFile(zip_path) as archive:
 for sub in range(123):
    name=f'sub{sub:03d}';member=f'Processed_data/{name}.pkl';info=archive.getinfo(member)
    source_rating=Path(r'C:\FACED dataset\Data')/name/'After_remarks.mat'
    rating_hash=hashlib.sha256(source_rating.read_bytes()).hexdigest()
    rating_dir=rating_root/name;rating_dir.mkdir(parents=True,exist_ok=True);shutil.copy2(source_rating,rating_dir/'After_remarks.mat')
    target=output/'data_processed'/'data_rPSD_FACED_V'/f'sub{sub}.pkl'
    marker=markers/f'{name}.json'
    identity={'subject':sub,'zip_member':member,'zip_crc32':info.CRC,'rating_sha256':rating_hash,'code_commit':source_commit,'parameters':vars(args)}
    if marker.exists() and target.exists() and json.loads(marker.read_text())==identity:
       print('EMT preparation already complete',sub,flush=True);continue
    FACEDZipInput(args).create_dataset([sub],split=1,sub_split=1,feature=1,band_pass_first=False)
    assert target.exists()
    marker.write_text(json.dumps(identity,indent=2),encoding='utf8')
    print('EMT prepared',sub+1,'/ 123 elapsed',round(time.perf_counter()-started,1),flush=True)
complete={'subjects':123,'source_commit':source_commit,'parameters':vars(args),'input_adapter':'zip stream for original Processed_data pickle; rating files copied without transformation','seconds':time.perf_counter()-started,'label_warning':'authors FACED.py uses rating score index 8 irrespective of label-type; index 8 is arousal under their documented rating order'}
(output/'preparation.json').write_text(json.dumps(complete,indent=2),encoding='utf8');print('EMT PREPROCESSING COMPLETE',flush=True)


In [ ]:
import os,subprocess,time,json
from pathlib import Path
art=Path(r'D:\EEG Experiment Artifacts\2026-11-02\graph_experiments')
source=art/'paper_sources'/'EmT'/'classification'
root=art/'emt_reproduction'
assert (root/'preparation.json').exists()
# The authors' default FACED code labels by rating score index 8 (arousal),
# even though the default label-type argument is V. Preserve this behavior.
finished=root/'author_run_complete.json'
if finished.exists():
    print('Completed author run:',finished.read_text(encoding='utf8'))
else:
    run_id=time.strftime('%Y%m%d-%H%M%S')
    run_dir=root/'runs'/run_id;run_dir.mkdir(parents=True,exist_ok=False)
    cmd=[sys.executable,'-u',str(source/'main-FACED.py'),
         '--ROOT',str(root),'--data-path',str(root/'input'),'--data-exist',
         '--save-path',str(run_dir/'save')]
    env=os.environ.copy();env['PYTHONPATH']=str(art/'paper_deps')+os.pathsep+env.get('PYTHONPATH','')
    print('Original author command:',cmd,flush=True)
    started=time.perf_counter()
    with (run_dir/'author.log').open('w',encoding='utf8',buffering=1) as log:
        process=subprocess.Popen(cmd,cwd=str(run_dir),env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf8',errors='replace',bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True);log.write(line)
        code=process.wait()
    if code:raise RuntimeError(f'Author run failed with exit code {code}; see {run_dir / "author.log"}')
    result_file=run_dir/'results_FACED.txt'
    assert result_file.exists()
    result={'run_dir':str(run_dir),'command':cmd,'seconds':time.perf_counter()-started,
            'source_commit':'3ea05dc90a915182ab3407191c287456f912316d','results_file':str(result_file),
            'note':'Authors code uses rating index 8 despite default V label; result is not comparable to fixed stimulus labels.'}
    finished.write_text(json.dumps(result,indent=2),encoding='utf8')
    print(result_file.read_text(encoding='utf8'))
